In [1]:
import random
import numpy as np
import torch
from torch.utils.data import Dataset
from pyfaidx import Fasta
import pyBigWig

In [24]:
# Dataset

fasta_path = "../data/hg38/hg38.fa"
bigwig_paths = [
    "../data/phylo/hg38.phyloP100way.bw", 
    # "../data/phyloP470.bw"
]
seq_len = 4096
chromosomes = [f"chr{i}" for i in range(1, 23)] + ["chrX"]
samples_per_epoch = 20000

In [3]:
DNA_VOCAB = {
    "A": 1,
    "C": 2,
    "G": 3,
    "T": 4,
    "N": 5,
}
PAD_ID = 0
MASK_ID = 6
UNK_ID = 7

In [4]:
fasta = Fasta(fasta_path, as_raw=True, sequence_always_upper=True)

In [7]:
chrom_sizes = {chrom: len(fasta[chrom]) for chrom in chromosomes}

In [13]:
total_size = sum(len(fasta[chrom]) for chrom in chromosomes)

print(f"Total size: {total_size:,} nucleotides")

Total size: 3,031,042,417 nucleotides


In [16]:
# Selected chromosome

chrom = random.choice(chromosomes)
chrom_len  = chrom_sizes[chrom]

start = random.randint(0, chrom_len - seq_len)
end = start + seq_len

chrom, start, end

('chr6', 85367716, 85371812)

In [17]:
seq = fasta[chrom][start:end]
seq

'TGATAAAGGGGATATCACCACTGATCCCACAGAAATACAAACTACCATCAGAGAATACCACAAACACCTCTACGCAAATAAACTAGAAAATCTAGAAGAAATGGATAAATTCCTGGACACATACACCATCCCAAGACTAAACCAGGAAGAAATTGAATCTCTGAATAGACCATTAACAGGCTCTGAAATTGTGGCAATAATCAATAGCTTACCAACCAAAAAGAGTCCAGGACCAGATGGATTCACAGCCGAATTCTACCAGAGGTACAAGGAGGAACTGGTACCATTCCTTCTGAAATTATTCCAATCAATAGAACAAGAGGGAATCCTCCCTAACTCATTTTATGAGAACAGCATCATCCTGATACCAAAGCCAGGCAGAGACACAACCAAAAAAGAGAATTTTAGACCAATATTCTTGATGAACATTGATGCAAAAATCCTCAAGAAAATAATGGCAAACCGAATCCAGCAGCAGATCAAAAAGCTTATCCACCATGATCAATTGGGCTTCATCCCTGGGATGCAAGGCTGGTTCAATATATGCAAATCAATAAATGTAATCCAGCATATAAACAGAACCAAAGACAAAAACCACATGATTATCTCCATAGATGCAGAAAAGGCCTTTGACAAAATTCAGCAACGCTTCATGCTAAAAACCGTCAATAAATTAGGTATTGATGGGATGAAAACATCTTTTCTTGTCAGGAAGACAGGCACCTTATAGGTAATCAATAAATGATGAGATGATGGTGGTGGTGGTGTCATTTATTTAAAGAGGGATAAAAGTGGATGGCTCAATTATTTTCACCTATTACAAGTCCTCCTCACCCCATCCAAATCTTATTCACATCAAATACTTTCTTCACAAGGACTTCTCACAGAATAATACAATTAAATATCATTAACACTGAAAGGAATGTTAGACGTTATTGACTCCAATGCCCTTGTTTTAACAGACGTGGAAAGAGTTAAAATGAACCTTCTCAGAGTTAA

In [25]:
bigwigs = [pyBigWig.open(p) for p in bigwig_paths]

In [33]:
for bw in bigwigs:
    scores = bw.values(chrom, start, end, numpy=True).astype(np.float32)
    print(scores)

[ 0.077  0.089  0.085 ... -0.343 -0.218 -0.063]


In [34]:
print(scores[0])

0.077
